In [16]:
# from huggingface_hub import snapshot_download

# snapshot_download(repo_id="mesolitica/haqkiem-TTS", repo_type="dataset", local_dir="./haqkiem-TTS")

In [9]:
# !wget https://www.7-zip.org/a/7z2501-linux-arm64.tar.xz
# !tar -xf 7z2501-linux-arm64.tar.xz
# !./7zz x haqkiem-TTS/wav.7z

In [31]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [30]:
import pandas as pd

df = pd.read_csv('haqkiem-TTS/metadata.csv', sep = '|', header = None).to_dict(orient = 'records')
for i in range(len(df)):
    df[i][1] = df[i][1].split(',,,,')[0]

In [36]:
def loop(rows):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    rows, _ = rows

    data = []
    base = 'haqkiem-TTS_audio'
    os.makedirs(base, exist_ok=True)

    for row in tqdm(rows):

        t = row[1].strip()
        if len(t) < 2:
            continue

        f_new = os.path.join('wav', row[0]) + '.wav'
        audio_filename = f_new.replace('/', '_').replace('.wav', '.mp3')
        audio_filename = os.path.join(base, audio_filename)
        audio_np, sr = sf.read(f_new)
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)
        if audio_np.shape[0] < 10000:
            continue
        sf.write(audio_filename, audio_np, sr)
        
        data.append({
            'audio_filename': audio_filename,
            'text': t,
            'speaker': "haqkiem-TTS"
        })
        
    return data

In [37]:
data = loop((df[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 20.03it/s]


In [38]:
data = multiprocessing(df, loop, cores = 20)

100%|██████████| 214/214 [00:14<00:00, 14.27it/s]


In [39]:
len(data)

4294

In [40]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'haqkiem-TTS_audio/wav_LJ001-000001.mp3',
 'text': 'Sultan Johor Sultan Ibrahim Iskandar selamat tiba di Lapangan Terbang Antarabangsa Senai malam tadi.',
 'speaker': 'haqkiem-TTS'}

In [41]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'haqkiem-TTS')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 380.71ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  300kB /  300kB, 1.50MB/s  
Processing Files (1 / 1): 100%|██████████|  300kB /  300kB,  749kB/s  
New Data Upload: 100%|██████████|  300kB /  300kB,  749kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.25 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/6ffa145e23b49ce90759292b720462df4b2aea22', commit_message='Upload dataset', commit_description='', oid='6ffa145e23b49ce90759292b720462df4b2aea22', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [42]:
audio_files = [d['audio_filename'] for d in data]

with open('haqkiem-TTS-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [45]:
folders = glob('haqkiem-TTS_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

haqkiem-TTS_audio
haqkiem-TTS_audio_neucodec


In [46]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('haqkiem-TTS_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 5.47MB / 5.47MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 5.47MB / 5.47MB,  0.00B/s  
New Data Upload: 100%|██████████| 5.47MB / 5.47MB,  0.00B/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  24%|██▍       | 53.5MB /  223MB,   ???B/s  
Processing Files (0 / 1):  87%|████████▋ |  195MB /  223MB,  707MB/s  
Processing Files (0 / 1):  99%|█████████▉|  221MB /  223MB,  419MB/s  
Processing Files (0 / 1):  99%|█████████▉|  221MB /  223MB,  280MB/s  
Processing Files (1 / 1): 100%|██████████|  223MB /  223MB,  169MB/s  
Processing Files (1 / 1): 100%|██████████|  223MB /  223MB,  141MB/s  
New Data Upload: 100%|██████████|  223MB /  223MB,  141MB/s  
